# Camera-domain PPP model comparison
Edit `metric_eval_config.py`, run the evaluator, and inspect the summary and plots.

In [1]:
from pathlib import Path
import h5py

from ppp_offline_metric_eval import load_python_config

cfg = load_python_config("metric_eval_config.py")

for model_name, h5_dir in cfg.H5_DIRS.items():
    print(f"\nModel: {model_name}")
    print(f"Directory: {h5_dir}")

    for path in sorted(Path(h5_dir).glob("rank*.h5")):
        try:
            with h5py.File(path, "r") as h5f:
                keys = list(h5f.keys())
                print(f"  OK:     {path.name}: {len(keys)} samples")
        except Exception as exc:
            print(f"  BROKEN: {path.name}: {exc}")


Model: 1069846_ppp_finals_unet_54k
Directory: /mnt/c/Users/uik04723/Repositories/gen_data_metric_finals_cam/1069846_ppp_finals_unet_54k_metric_eval_CamDom/20260728_023617/metric_eval_data/test
  OK:     rank00.h5: 75 samples
  OK:     rank01.h5: 75 samples
  OK:     rank02.h5: 75 samples
  OK:     rank03.h5: 75 samples


In [2]:
from ppp_offline_metric_eval import run_from_config

df_samples, df_summary = run_from_config("metric_eval_config.py")
df_summary

1069846_ppp_finals_unet_54k: 300 samples from /mnt/c/Users/uik04723/Repositories/gen_data_metric_finals_cam/1069846_ppp_finals_unet_54k_metric_eval_CamDom/20260728_023617/metric_eval_data/test
  1069846_ppp_finals_unet_54k: 1/300
  1069846_ppp_finals_unet_54k: 25/300
  1069846_ppp_finals_unet_54k: 50/300
  1069846_ppp_finals_unet_54k: 75/300
  1069846_ppp_finals_unet_54k: 100/300
  1069846_ppp_finals_unet_54k: 125/300
  1069846_ppp_finals_unet_54k: 150/300
  1069846_ppp_finals_unet_54k: 175/300
  1069846_ppp_finals_unet_54k: 200/300
  1069846_ppp_finals_unet_54k: 225/300
  1069846_ppp_finals_unet_54k: 250/300
  1069846_ppp_finals_unet_54k: 275/300
  1069846_ppp_finals_unet_54k: 300/300

Model comparison summary:
                      model  n_samples  n_samples_ok  evaluation_ok_fraction  expected_count_mean  gt_count_mean  count_bias  count_error_std  count_mae  count_rmse  count_ratio_mean  likelihood_optimal_score_mean  likelihood_optimal_score_std  likelihood_optimal_score_median  

,model,n_samples,n_samples_ok,evaluation_ok_fraction,expected_count_mean,gt_count_mean,count_bias,count_error_std,count_mae,count_rmse,...,depth_mae_global_bootstrap_se,depth_mae_global_bootstrap_n_valid,velocity_mae_global_ci95_low,velocity_mae_global_ci95_high,velocity_mae_global_bootstrap_se,velocity_mae_global_bootstrap_n_valid,rcs_mae_global_ci95_low,rcs_mae_global_ci95_high,rcs_mae_global_bootstrap_se,rcs_mae_global_bootstrap_n_valid
0,1069846_ppp_finals_unet_54k,300,300,1.0,165.679799,222.576667,-56.896868,48.578067,61.449421,74.813649,...,0.240033,5000,0.358056,0.450585,0.023643,5000,5.266293,5.565833,0.07712,5000


In [4]:
df_summary.columns

Index(['model', 'n_samples', 'n_samples_ok', 'evaluation_ok_fraction',
       'expected_count_mean', 'gt_count_mean', 'count_bias', 'count_error_std',
       'count_mae', 'count_rmse', 'count_ratio_mean',
       'likelihood_optimal_score_mean', 'likelihood_optimal_score_std',
       'likelihood_optimal_score_median', 'likelihood_optimal_sigma_px_mean',
       'likelihood_optimal_sigma_px_std', 'likelihood_score_sigma0_mean',
       'depth_mae_frame_mean', 'depth_rmse_frame_mean',
       'depth_bias_frame_mean', 'depth_mae_global', 'depth_rmse_global',
       'depth_bias_global', 'depth_n_points_global',
       'depth_abs_rel_error_global', 'velocity_mae_frame_mean',
       'velocity_rmse_frame_mean', 'velocity_bias_frame_mean',
       'velocity_mae_global', 'velocity_rmse_global', 'velocity_bias_global',
       'velocity_n_points_global', 'rcs_mae_frame_mean', 'rcs_rmse_frame_mean',
       'rcs_bias_frame_mean', 'rcs_mae_global', 'rcs_rmse_global',
       'rcs_bias_global', 'rcs_n_poin

In [5]:
main_columns = [
    "model",
    "count_rmse",
    "likelihood_optimal_score_mean",
    "depth_mae_global",
    "velocity_mae_global",
    "rcs_mae_global",
]
display(df_summary[[c for c in main_columns if c in df_summary.columns]].round(4))

,model,count_rmse,likelihood_optimal_score_mean,depth_mae_global,velocity_mae_global,rcs_mae_global
0,1069846_ppp_finals_unet_54k,74.8136,7.1458,7.1838,0.4018,5.4117


In [6]:
import pandas as pd


camera_metrics = [
    "count_rmse",
    "likelihood_optimal_score_mean",
    "depth_mae_global",
    "velocity_mae_global",
    "rcs_mae_global",
]

camera_metric_labels = {
    "count_rmse": "Count RMSE ↓",
    "likelihood_optimal_score_mean": "Optimal logL score ↑",
    "depth_mae_global": "Depth MAE [m] ↓",
    "velocity_mae_global": "Velocity MAE [m/s] ↓",
    "rcs_mae_global": "RCS MAE [dBsm] ↓",
}


def format_estimate_with_ci(
    estimate: float,
    ci_low: float,
    ci_high: float,
    decimals: int = 3,
) -> str:
    """Format a point estimate followed by its 95% confidence interval."""
    if pd.isna(estimate):
        return "—"

    if pd.isna(ci_low) or pd.isna(ci_high):
        return f"{estimate:.{decimals}f}"

    return (
        f"{estimate:.{decimals}f} "
        f"[{ci_low:.{decimals}f}, {ci_high:.{decimals}f}]"
    )


required_columns = [
    "model",
    *camera_metrics,
    *[
        column
        for metric in camera_metrics
        for column in (
            f"{metric}_ci95_low",
            f"{metric}_ci95_high",
        )
    ],
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_summary.columns
]

if missing_columns:
    raise KeyError(
        "Bootstrap confidence-interval columns are missing. "
        "Rerun the evaluator with BOOTSTRAP enabled.\n"
        f"Missing columns: {missing_columns}"
    )


camera_table = df_summary[required_columns].copy()

# Sort before converting the numerical estimates to strings.
camera_table = camera_table.sort_values(
    by="count_rmse",
    ascending=True,
).reset_index(drop=True)

for metric in camera_metrics:
    camera_table[metric] = camera_table.apply(
        lambda row: format_estimate_with_ci(
            estimate=row[metric],
            ci_low=row[f"{metric}_ci95_low"],
            ci_high=row[f"{metric}_ci95_high"],
            decimals=3,
        ),
        axis=1,
    )

camera_table = camera_table[
    ["model", *camera_metrics]
].rename(
    columns={
        "model": "Model",
        **camera_metric_labels,
    }
)

display(camera_table)

,Model,Count RMSE ↓,Optimal logL score ↑,Depth MAE [m] ↓,Velocity MAE [m/s] ↓,RCS MAE [dBsm] ↓
0,1069846_ppp_finals_unet_54k,"74.814 [69.109, 80.194]","7.146 [7.047, 7.240]","7.184 [6.729, 7.664]","0.402 [0.358, 0.451]","5.412 [5.266, 5.566]"


In [ ]:
from ppp_plotting import plot_expected_count_scatter

plot_expected_count_scatter(
    df_samples,
    output_dir="camera_metric_results/plots",
);

In [ ]:
from ppp_plotting import plot_metric_boxplots

plot_metric_boxplots(
    df_samples,
    [
        "likelihood_optimal_score",
        "depth_mae",
        "velocity_mae",
        "rcs_mae",
    ],
    output_dir="camera_metric_results/plots",
);

In [ ]:
# Verify that every model contains exactly the same sample IDs.
sample_sets = {
    model: set(group["sample_uid"])
    for model, group in df_samples.groupby("model")
}
{model: len(values) for model, values in sample_sets.items()}